# Ash Gourd Leaf Disease Classification (Kaggle)
Dataset: [Ash Gourd Leaf Healthy and Disease Dataset](https://data.mendeley.com/datasets/zj4th6xvdp/2) (5 classes: Healthy, Aphid, Downy Mildew, Leaf Curl, Leaf Miner)

**Before running:**
1. Settings (right panel) → Accelerator → **GPU T4 x2** or **GPU P100**.
2. Right panel → **Add Input** → add your uploaded Ash Gourd dataset.
3. Run all cells.

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
import os
import zipfile
import matplotlib.pyplot as plt

## Step 1: Find the dataset in /kaggle/input
Looks for the class folders (Healthy, Aphid, ...). If the dataset is still a zip, it extracts it to /kaggle/working.

In [ ]:
def is_dataset_root(dirs):
    names = [d.lower() for d in dirs]
    return any('healthy' in n for n in names) and any('aphid' in n for n in names)

data_folder = None
zip_file = None

for root, dirs, files_ in os.walk('/kaggle/input'):
    if is_dataset_root(dirs):
        data_folder = root
        break
    for f in files_:
        if f.lower().endswith('.zip') and zip_file is None:
            zip_file = os.path.join(root, f)

# If only a zip was found, extract it
if data_folder is None and zip_file is not None:
    extract_dir = '/kaggle/working/ash_gourd'
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(zip_file, 'r') as z:
        z.extractall(extract_dir)
    for root, dirs, files_ in os.walk(extract_dir):
        if is_dataset_root(dirs):
            data_folder = root
            break

if data_folder is None:
    raise FileNotFoundError('Dataset not found. Use Add Input (right panel) to attach your Ash Gourd dataset.')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
print('Dataset folder:', data_folder)

## Step 2: Load images and split into train / validation (80/20 per class)

In [ ]:
IMG_SIZE = 32   # same as the sample code; try 64 for better accuracy (leaf details)
NUM_CLASSES = 5

class_names = sorted([
    folder for folder in os.listdir(data_folder)
    if os.path.isdir(os.path.join(data_folder, folder))
])

print('Classes:', class_names)
print('Number of classes:', len(class_names))
assert len(class_names) == NUM_CLASSES, f'Expected {NUM_CLASSES} classes, found {len(class_names)}. Check the folder printed above.'

def list_images(folder):
    paths = []
    for root, _, files_ in os.walk(folder):
        for f in files_:
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                paths.append(os.path.join(root, f))
    return sorted(paths)

tr_images, tr_targets = [], []
val_images, val_targets = [], []

np.random.seed(42)

for label, class_name in enumerate(class_names):
    class_folder = os.path.join(data_folder, class_name)
    image_files = list_images(class_folder)
    np.random.shuffle(image_files)
    split_index = int(len(image_files) * 0.8)
    print(f'{class_name}: {len(image_files)} images')

    for i, image_path in enumerate(image_files):
        with Image.open(image_path) as image:
            image = np.array(image.convert('RGB').resize((IMG_SIZE, IMG_SIZE)))

        if i < split_index:
            tr_images.append(image)
            tr_targets.append(label)
        else:
            val_images.append(image)
            val_targets.append(label)

tr_images = np.array(tr_images, dtype=np.uint8)
tr_targets = np.array(tr_targets, dtype=np.int64)
val_images = np.array(val_images, dtype=np.uint8)
val_targets = np.array(val_targets, dtype=np.int64)

print('Training image shape:', tr_images.shape)
print('Validation image shape:', val_images.shape)
print('Training labels:', tr_targets.shape)
print('Validation labels:', val_targets.shape)

## Step 3: Hyperparameters

In [ ]:
batch_size_no = 32
learning_rate = 1e-3
epoch_number = 30

## Step 4: Dataset and DataLoader

In [ ]:
class AshGourdLeaf(Dataset):
    def __init__(self, x, y):
        x = torch.tensor(x)        # N, H, W, C
        x = x.float() / 255        # Normalization: 0..255 -> 0..1
        x = x.permute(0, 3, 1, 2)  # N, C, H, W
        x = x.reshape(-1, 3, IMG_SIZE, IMG_SIZE)

        y = torch.tensor(y)
        self.x, self.y = x, y

    def __getitem__(self, ix):
        return self.x[ix], self.y[ix]

    def __len__(self):
        return len(self.x)


def get_data():
    train = AshGourdLeaf(tr_images, tr_targets)
    trn_dl = DataLoader(train, batch_size=batch_size_no, shuffle=True)
    val = AshGourdLeaf(val_images, val_targets)
    val_dl = DataLoader(val, batch_size=batch_size_no, shuffle=False)
    return trn_dl, val_dl

In [ ]:
trn_dl, val_dl = get_data()
print('Number of training batches:', len(trn_dl))
print('Number of validation batches:', len(val_dl))

## Step 5: Model

In [ ]:
from torch.optim import Adam

def get_model():
    final_size = IMG_SIZE // 8   # three MaxPool2d(2) layers
    model = nn.Sequential(
        # 1st conv
        nn.Conv2d(3, 40, 3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
        # 2nd conv
        nn.Conv2d(40, 65, 3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
        # 3rd conv
        nn.Conv2d(65, 70, 3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
        # Classifier
        nn.Flatten(),
        nn.Linear(70 * final_size * final_size, 256),
        nn.ReLU(),
        nn.Linear(256, NUM_CLASSES)
    ).to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = Adam(model.parameters(), lr=learning_rate)
    return model, loss_fn, optimizer

## Step 6: Train and validation functions

In [ ]:
def train_one_epoch(data_loader, model, loss_fn, optimizer):
    model.train()
    epoch_losses, epoch_accuracies = [], []
    for x, y in data_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        prediction = model(x)
        batch_loss = loss_fn(prediction, y)
        batch_loss.backward()
        optimizer.step()

        epoch_losses.append(batch_loss.item())
        is_correct = (prediction.argmax(-1) == y).tolist()
        epoch_accuracies.extend(is_correct)
    return np.mean(epoch_losses), np.mean(epoch_accuracies)

In [ ]:
@torch.no_grad()
def validation(data_loader, model, loss_fn):
    model.eval()
    val_losses, val_accuracies = [], []
    for x, y in data_loader:
        x, y = x.to(device), y.to(device)
        prediction = model(x)
        val_losses.append(loss_fn(prediction, y).item())
        is_correct = (prediction.argmax(-1) == y).tolist()
        val_accuracies.extend(is_correct)
    return np.mean(val_losses), np.mean(val_accuracies)

## Step 7: Build the model

In [ ]:
trn_dl, val_dl = get_data()
model, loss_fn, optimizer = get_model()
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

## Step 8: Train

In [ ]:
from torch import optim
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    factor=0.5,
    patience=0,
    threshold=0.001,
    min_lr=1e-5,
    threshold_mode='abs'
)

train_losses, train_accuracies = [], []
val_losses, val_accuracies = [], []

for epoch in range(epoch_number):
    epoch_loss, epoch_accuracy = train_one_epoch(trn_dl, model, loss_fn, optimizer)
    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_accuracy)

    v_loss, v_acc = validation(val_dl, model, loss_fn)
    val_losses.append(v_loss)
    val_accuracies.append(v_acc)

    scheduler.step(v_loss)

    print(f'Epoch [{epoch + 1}/{epoch_number}]'
          f' | Train Loss: {epoch_loss:.4f} | Train Accuracy: {epoch_accuracy*100:.2f}%'
          f' | Val Loss: {v_loss:.4f} | Val Accuracy: {v_acc*100:.2f}%')

## Step 9: Plot results

In [ ]:
epochs = range(1, epoch_number + 1)
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, train_losses, label='Train')
plt.plot(epochs, val_losses, label='Validation')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Loss'); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, train_accuracies, label='Train')
plt.plot(epochs, val_accuracies, label='Validation')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.title('Accuracy'); plt.legend()

plt.tight_layout()
plt.show()